In [4]:
import uproot
import numpy as np
import pandas as pd
from pathlib import Path

TREE = "AnalysisMiniTree"
CHANNEL = "1l2tau"      # o "2l2tau"
RUN = "run2"            # o "run3"

# Resolvemos la carpeta base sin importar el cwd del kernel
CANDIDATE_BASES = [
    Path("PPSSP_2026"),
    Path("Esteban/PPSSP_2026"),
    Path("/home/est3ban3/IFJ_PAN_PROJECT/MVA-Project/Esteban/PPSSP_2026"),
]
BASE = next((b for b in CANDIDATE_BASES if (b / CHANNEL / RUN).exists()), None)
if BASE is None:
    raise FileNotFoundError(f"No encuentro PPSSP_2026/{CHANNEL}/{RUN} en {CANDIDATE_BASES}")

# proceso : nombre de archivo real (sin .root)
SAMPLES = {
    "signal_ggF": "signal_ggF",
    "signal_VBF": "signal_VBF",
    "Diboson":    "diboson",
    "SingleH":    "singleH",
    "tops":       "tops",
    "ttbar":      "ttbar",
    "Vgamma":     "Vgamma",
    "VVV":        "VVV",
    "Wjets":      "Wjets",
    "Zjets":      "Zjets",
}

SIGNALS = {"signal_ggF", "signal_VBF"}


def preselection(arrays, channel):
    """Máscara booleana con los cortes del README."""
    if channel == "1l2tau":
        return (arrays["n_b_jet"] == 0) & (arrays["n_jet"] >= 2)
    elif channel == "2l2tau":
        return (
            (arrays["n_b_jet"] == 0)
            & (arrays["l1_charge"] * arrays["l2_charge"] < 0)   # <-- confirmar nombres
            & (arrays["mZ_cut"] > 0)                            # <-- confirmar qué es
        )
    raise ValueError(channel)


rows = []
for proc, fname in SAMPLES.items():
    path = BASE / CHANNEL / RUN / f"{fname}.root"
    if not path.exists():
        print(f"  [skip] no existe: {path}")
        continue

    tree = uproot.open({str(path): TREE})

    # 'weight'       = σ · L_campaña / Σw_gen  (constante por DSID+campaña, normalización)
    # 'weight_final' = peso del evento del generador (Sherpa NLO -> tiene negativos)
    # peso físico    = weight * weight_final
    needed = ["weight", "weight_final", "n_b_jet", "n_jet", "l1_charge", "l2_charge", "mZ_cut"]
    available = [b for b in needed if b in tree.keys()]
    arr = tree.arrays(available, library="np")

    w_phys = arr["weight"] * arr["weight_final"]
    mask = preselection(arr, CHANNEL)
    w_sel = w_phys[mask]

    rows.append({
        "process":    proc,
        "is_signal":  proc in SIGNALS,
        "N_mc":       len(w_phys),           # eventos MC en el archivo (ya skimmed a pass1l2tau)
        "yield_skim": w_phys.sum(),          # yield antes de los cortes del README
        "N_presel":   int(mask.sum()),
        "yield":      w_sel.sum(),           # yield tras preselección (n_b_jet==0 & n_jet>=2)
        "sum_abs_w":  np.abs(w_sel).sum(),
        "frac_neg":   (w_sel < 0).mean() if len(w_sel) else 0.0,
    })

df = pd.DataFrame(rows)

# fracción del background total (tras preselección)
bkg_total = df.loc[~df.is_signal, "yield"].sum()
sig_total = df.loc[df.is_signal, "yield"].sum()
df["pct_of_bkg"] = np.where(~df.is_signal, 100 * df["yield"] / bkg_total, np.nan)

df = df.sort_values(["is_signal", "yield"], ascending=[True, False])

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
cols = ["process", "N_mc", "yield_skim", "N_presel", "yield", "pct_of_bkg", "frac_neg"]
print(df[cols].to_string(index=False))
print(f"\nTotal background (presel): {bkg_total:,.3f}")
print(f"Total signal     (presel): {sig_total:,.3f}")
print(f"S/B  = {sig_total/bkg_total:.2e}")
print(f"S/√B = {sig_total/np.sqrt(bkg_total):.4f}")


   process   N_mc  yield_skim  N_presel     yield  pct_of_bkg  frac_neg
     Wjets  74455  39,768.937     31272 5,698.655      49.695     0.158
     Zjets 308295  20,845.893    141863 3,582.158      31.238     0.154
     ttbar  58930   7,962.950      7335   994.253       8.670     0.004
   Diboson 621842   2,379.632    312600   730.367       6.369     0.062
    Vgamma  53058   1,400.002     21510   276.489       2.411     0.079
      tops 512118     914.310     62238   120.914       1.054     0.363
   SingleH 130938     333.658     11124    59.853       0.522     0.005
       VVV  43008      13.026     14028     4.598       0.040     0.002
signal_ggF 133971       3.381     67075     1.757         NaN     0.060
signal_VBF  47257       0.150     21639     0.068         NaN     0.001

Total background (presel): 11,467.287
Total signal     (presel): 1.824
S/B  = 1.59e-04
S/√B = 0.0170
